# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Institución](https://img.shields.io/badge/INSTITUCIÓN-TECNOLÓGICO_DE_MONTERREY-0036a3?style=for-the-badge)](https://tec.mx)
[![Programa](https://img.shields.io/badge/PROGRAMA-MNA-0036a3?style=for-the-badge)](https://tec.mx)
[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![NumPy](https://img.shields.io/badge/NumPy-013243?style=flat-square&logo=numpy&logoColor=white)](https://numpy.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5C3EE8?style=flat-square&logo=opencv&logoColor=white)](https://opencv.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

---

## **<font color="#0036a3">Semana 5: Avance 2 — Ingeniería de Características</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**
**Materia:** Proyecto Integrador — TC5035.10

> **Título:** La luz que opaca: Detección y supresión de reflejos especulares en imágenes endoscópicas.

---

### **Equipo Docente**
* **Dra. Grettel Barceló Alonso** | Profesora Titular
* **Dr. Luis Eduardo Falcón Morales** | Profesor Titular
* **Mtra. Verónica Sandra Guzmán de Valle** | Profesora Asistente

**Patrocinador:** Dr. Gerardo Camacho

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="width:100%; border:none; border-collapse:collapse;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://ui-avatars.com/api/?name=Elda+Morales&size=140&background=0036a3&color=fff&rounded=true" width="140px" style="border-radius:10px;">
      <br>
      <h4>Elda Cristina Morales Sánchez de la Barquera</h4>
      <code>A00449074</code><br>
      <small>MNA Student</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://ui-avatars.com/api/?name=Maria+Gutierrez&size=140&background=0036a3&color=fff&rounded=true" width="140px" style="border-radius:10px;">
      <br>
      <h4>María Paula Gutiérrez Cervantes</h4>
      <code>A01747706</code><br>
      <small>MNA Student</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc.jpg" width="160px" height="160px" style="border-radius:50%; object-fit:cover;">
      <br>
      <h4>José Manuel Toral Cruz</h4>
      <code>A01122243</code><br>
      <small>MNA Student</small>
    </td>
  </tr>
</table>

---

### **<font color="#E0A800">Información de Entrega</font>**
* **Actividad:** Avance 2. Ingeniería de características.
* **Fecha Límite:** 25 de mayo de 2026
* **Framework de Calidad:** CRISP-ML(Q)
* **Objetivos Clave:**
    * **Feature Engineering:** Construcción de representación tabular derivada de imágenes endoscópicas.
    * **Transformaciones:** Cap a 150 mm, log(Z+1), normalización min-max.
    * **Selección:** Correlación de Pearson, umbral de varianza, PCA geométrico.

---

## FE 01 — Ingeniería de características para estimación de profundidad endoscópica

**¿Qué hace este notebook?**

El dataset SCARED no es tabular: cada observación es un par imagen RGB + mapa de profundidad TIFF. Esta fase de ingeniería de características construye una **representación derivada** que permite aplicar técnicas de diagnóstico estadístico, selección de features y análisis de calidad de datos antes de entrenar modelos de profundidad profunda.

El pipeline produce dos tipos de artefactos complementarios:
- **Dataframe de 20 filas × N features** (una fila por keyframe): útil para diagnóstico, selección y análisis de correlaciones.
- **Pipeline de preprocesamiento de tensores** (cap, log, normalización): se aplicará en el training loop de los modelos NeXF, EndoGaussian y EndoDepthAndMotion.


---

## Sección 0 — Setup y carga de datos


In [ ]:
!pip install tifffile -q

In [ ]:
import zipfile
import io
import tempfile
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import tifffile

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

In [ ]:
DATASETS = {
    "dataset_1": {
        "zip":       "D:/scared_raw/dataset_1.zip",
        "keyframes": ["keyframe_1", "keyframe_2", "keyframe_3",
                      "keyframe_4", "keyframe_5"],
        "split":     "train",
        "cerdo_id":  "cerdo_A",
    },
    "dataset_2": {
        "zip":       "D:/scared_raw/dataset_2.zip",
        "keyframes": ["keyframe_1", "keyframe_2", "keyframe_3",
                      "keyframe_4", "keyframe_5"],
        "split":     "train",
        "cerdo_id":  "cerdo_A",
    },
    "dataset_8": {
        "zip":       "D:/scared_raw/dataset_8.zip",
        "keyframes": ["keyframe_0", "keyframe_1", "keyframe_2",
                      "keyframe_3", "keyframe_4"],
        "split":     "test",
        "cerdo_id":  "cerdo_C",
    },
    "dataset_9": {
        "zip":       "D:/scared_raw/dataset_9.zip",
        "keyframes": ["keyframe_0", "keyframe_1", "keyframe_2",
                      "keyframe_3", "keyframe_4"],
        "split":     "test",
        "cerdo_id":  "cerdo_C",
    },
}

### Verificación de archivos zip

Se confirma que cada zip existe en disco y se reporta su tamaño antes de iniciar la extracción.

In [ ]:
from pathlib import Path

for ds_id, info in DATASETS.items():
    p = Path(info["zip"])
    if p.exists():
        size_gb = p.stat().st_size / (1024 ** 3)
        print(f"✓ {ds_id}: {p.name}  ({size_gb:.2f} GB)")
    else:
        print(f"✗ {ds_id}: NO ENCONTRADO — {p}")

---

## Sección 1 — Construcción de características

El dataset SCARED no es tabular en su forma nativa: cada keyframe es una carpeta con una imagen RGB, un mapa de profundidad TIFF, una nube de puntos OBJ y un video MP4. Para aplicar técnicas estadísticas de selección y diagnóstico es necesario construir primero una **representación derivada** en forma de tabla.

Este notebook implementa dos flujos paralelos:

1. **Dataframe por keyframe**: una fila por combinación dataset/keyframe con features estadísticas extraídas de la imagen y del mapa de profundidad. Se usa para diagnóstico, correlación y selección.
2. **Pipeline de tensores**: las transformaciones (cap, log, normalización) se definen aquí para ser reutilizadas en el training loop de los modelos profundos.

### 1a. Extracción de features por keyframe

**Features extraídas:**

Del canal Z del TIFF (convertir ceros a NaN, cap a 150 mm):
- `depth_median`, `depth_p5`, `depth_p95`, `depth_range`, `depth_valid_pct`

De `Left_Image.png` (luminancia ITU-R BT.601: Y = 0.299R + 0.587G + 0.114B):
- `lum_mean`, `lum_std`, `sat_pct`, `laplacian_var`, `r_mean`, `g_mean`, `b_mean`

**Fuente:** Allan et al. (2021). *Stereo Correspondence and Reconstruction of Endoscopic Data Challenge*. arXiv:2101.01133.

In [ ]:
def load_depth_and_image(zf, ds_id, kf_id):
    """Carga TIFF y Left_Image.png desde un ZipFile abierto. Devuelve (Z, img_rgb)."""
    prefix = f"{ds_id}/{kf_id}/"

    # ── TIFF ──────────────────────────────────────────────────────────────
    tiff_name = prefix + "left_depth_map.tiff"
    with zf.open(tiff_name) as f:
        buf = io.BytesIO(f.read())
    data = tifffile.imread(buf)          # float32, shape (H, W, 3)
    Z = data[..., 2].astype(np.float32) # canal Z en mm
    Z[Z <= 0] = np.nan                  # (0,0,0) = sin medición

    # ── RGB ───────────────────────────────────────────────────────────────
    img_name = prefix + "Left_Image.png"
    with zf.open(img_name) as f:
        buf = io.BytesIO(f.read())
    img_arr = np.frombuffer(buf.getvalue(), np.uint8)
    img_rgb = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)

    return Z, img_rgb


def extract_features(Z, img_rgb):
    """Extrae features estadísticas de Z (mm, con NaN) e imagen RGB."""
    # ── Profundidad ───────────────────────────────────────────────────────
    Z_cap = np.where((Z > 0) & (Z < 150), Z, np.nan)
    depth_valid_pct  = float(np.sum(~np.isnan(Z_cap)) / Z.size * 100)
    depth_median     = float(np.nanmedian(Z_cap))
    depth_p5         = float(np.nanpercentile(Z_cap, 5))
    depth_p95        = float(np.nanpercentile(Z_cap, 95))
    depth_range      = depth_p95 - depth_p5

    # ── Imagen ────────────────────────────────────────────────────────────
    R = img_rgb[:, :, 0].astype(np.float32)
    G = img_rgb[:, :, 1].astype(np.float32)
    B = img_rgb[:, :, 2].astype(np.float32)
    lum   = 0.299 * R + 0.587 * G + 0.114 * B
    lum_mean = float(lum.mean())
    lum_std  = float(lum.std())
    sat_mask = (img_rgb > 240).any(axis=2)
    sat_pct  = float(sat_mask.sum() / img_rgb[:, :, 0].size * 100)
    gray     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    lap_var  = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    r_mean   = float(R.mean())
    g_mean   = float(G.mean())
    b_mean   = float(B.mean())

    return dict(
        depth_median    = depth_median,
        depth_p5        = depth_p5,
        depth_p95       = depth_p95,
        depth_range     = depth_range,
        depth_valid_pct = depth_valid_pct,
        lum_mean        = lum_mean,
        lum_std         = lum_std,
        sat_pct         = sat_pct,
        laplacian_var   = lap_var,
        r_mean          = r_mean,
        g_mean          = g_mean,
        b_mean          = b_mean,
    )


rows = []
for ds_id, info in DATASETS.items():
    print(f"Procesando {ds_id}...")
    with zipfile.ZipFile(info["zip"], "r") as zf:
        for kf_id in info["keyframes"]:
            Z, img_rgb = load_depth_and_image(zf, ds_id, kf_id)
            feats = extract_features(Z, img_rgb)
            feats.update(
                dataset_id  = ds_id,
                keyframe_id = kf_id,
                split       = info["split"],
                cerdo_id    = info["cerdo_id"],
            )
            rows.append(feats)
            print(f"  ✓ {kf_id}  depth_valid={feats['depth_valid_pct']:.1f}%"
                  f"  lum_mean={feats['lum_mean']:.1f}")

df = pd.DataFrame(rows)
META_COLS = ["dataset_id", "keyframe_id", "split", "cerdo_id"]
FEAT_COLS = [c for c in df.columns if c not in META_COLS]

print(f"\nDataframe construido: {df.shape}")
df.head(20)

In [ ]:
df[FEAT_COLS].describe().round(3)

### 1b. Características derivadas

Las siguientes features de segundo orden capturan relaciones entre variables primarias que tienen interpretación física directa en el dominio endoscópico.

In [ ]:
# rb_balance: ratio rojo/azul
# El tejido abdominal porcino tiene tonos rojizos; un valor alto indica
# tejido sano, uno bajo puede indicar sangrado o cambio de iluminación.
df["rb_balance"] = df["r_mean"] / (df["b_mean"] + 1e-6)

# log_depth_median: versión transformada de la profundidad mediana.
# Justificada en la sección 2b por el sesgo positivo de la distribución.
df["log_depth_median"] = np.log1p(df["depth_median"])

# texture_depth_ratio: riqueza de textura relativa a variación geométrica.
# Alto = tejido plano texturado; bajo = geometría compleja con poca textura.
df["texture_depth_ratio"] = df["laplacian_var"] / (df["depth_range"] + 1e-6)

# lum_contrast_ratio: relación señal-ruido de la iluminación.
# Bajo = escena con alto contraste donde la photometric loss es menos fiable.
df["lum_contrast_ratio"] = df["lum_mean"] / (df["lum_std"] + 1e-6)

FEAT_COLS = [c for c in df.columns if c not in META_COLS]
print(f"Features totales tras derivadas: {len(FEAT_COLS)}")
df[["dataset_id", "keyframe_id"] + ["rb_balance", "log_depth_median",
   "texture_depth_ratio", "lum_contrast_ratio"]].head(20)

### 1c. Máscara de viñeteado gaussiano (feature espacial)

El EDA de iluminación (notebook 01) mostró que los píxeles inválidos del GT no son aleatorios: siguen el patrón de viñeteado circular del endoscopio. La región central tiene más cobertura GT que la periferia. Esto permite construir una feature espacial — `vignette_score` — que actúa como proxy de la calidad de la medición de profundidad por keyframe.

La máscara se ajusta con una gaussiana 1D sobre el perfil radial de validez, siguiendo la metodología de Godard et al. (2019) para modelar el prior espacial de confianza en mapas de profundidad.

In [ ]:
# ── Parámetros espaciales ────────────────────────────────────────────────
CX, CY = 640, 512   # centro geométrico (imagen 1280×1024)

def gaussian_1d(r, A, sigma, r0):
    return A * np.exp(-((r - r0) ** 2) / (2 * sigma ** 2))


masks_ds1 = []
with zipfile.ZipFile(DATASETS["dataset_1"]["zip"], "r") as zf:
    for kf_id in DATASETS["dataset_1"]["keyframes"]:
        Z, _ = load_depth_and_image(zf, "dataset_1", kf_id)
        valid = ((Z > 0) & (Z < 150)).astype(np.float32)
        masks_ds1.append(valid)

mean_mask = np.mean(masks_ds1, axis=0)   # (H, W) mapa de validez promedio

# ── Perfil radial ─────────────────────────────────────────────────────────
H, W = mean_mask.shape
yy, xx = np.mgrid[0:H, 0:W]
R_map = np.sqrt((xx - CX) ** 2 + (yy - CY) ** 2).astype(int)

r_max = int(R_map.max())
r_vals, v_vals = [], []
for r in range(0, r_max, 2):
    idx = (R_map == r)
    if idx.sum() > 50:
        r_vals.append(r)
        v_vals.append(mean_mask[idx].mean())
r_vals = np.array(r_vals, dtype=float)
v_vals = np.array(v_vals, dtype=float)

# ── Ajuste gaussiano ──────────────────────────────────────────────────────
try:
    popt, _ = curve_fit(gaussian_1d, r_vals, v_vals,
                        p0=[1.0, 300, 0], maxfev=5000)
except RuntimeError:
    popt = [1.0, 300, 0]

r_fit = np.linspace(0, r_max, 400)
v_fit = gaussian_1d(r_fit, *popt)

# ── Figura ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(mean_mask, cmap="viridis", vmin=0, vmax=1)
axes[0].scatter([CX], [CY], c="red", s=60, zorder=5, label="Centro geométrico")
axes[0].set_title("Mapa de validez promedio — dataset_1")
axes[0].set_xlabel("Columna (px)")
axes[0].set_ylabel("Fila (px)")
axes[0].legend()
plt.colorbar(im, ax=axes[0], label="Fracción de keyframes con GT válido")

axes[1].scatter(r_vals, v_vals, s=10, alpha=0.6, label="Perfil radial empírico")
axes[1].plot(r_fit, v_fit, "r-", lw=2, label=f"Gaussiana ajustada
(σ={popt[1]:.0f} px)")
axes[1].set_title("Perfil radial de validez del GT")
axes[1].set_xlabel("Distancia al centro geométrico (px)")
axes[1].set_ylabel("Fracción de validez promedio")
axes[1].legend()

for ax in axes:
    ax.text(0.5, -0.12, "Fuente: elaboración propia con base en Allan et al. (2021)",
            transform=ax.transAxes, fontsize=8, ha="center", color="gray")

plt.tight_layout()
plt.show()
print(f"Parámetros gaussianos — A={popt[0]:.3f}, σ={popt[1]:.1f} px, r₀={popt[2]:.1f} px")

In [ ]:
# ── vignette_score por keyframe ───────────────────────────────────────────
# 25% central: columnas [CX-W//8, CX+W//8] × filas [CY-H//8, CY+H//8]
h_q, w_q = H // 8, W // 8
center_slice = mean_mask[CY - h_q:CY + h_q, CX - w_q:CX + w_q]
vignette_global = float(center_slice.mean())
print(f"Validez en 25% central (global dataset_1): {vignette_global:.3f}")

# Por keyframe individual
vignette_scores = {}
for i, kf_id in enumerate(DATASETS["dataset_1"]["keyframes"]):
    cs = masks_ds1[i][CY - h_q:CY + h_q, CX - w_q:CX + w_q]
    vignette_scores[("dataset_1", kf_id)] = float(cs.mean())

# Para los demás datasets usar el mapa global de dataset_1 como referencia
for ds_id, info in DATASETS.items():
    if ds_id == "dataset_1":
        continue
    with zipfile.ZipFile(info["zip"], "r") as zf:
        for kf_id in info["keyframes"]:
            Z, _ = load_depth_and_image(zf, ds_id, kf_id)
            valid = ((Z > 0) & (Z < 150)).astype(np.float32)
            cs = valid[CY - h_q:CY + h_q, CX - w_q:CX + w_q]
            vignette_scores[(ds_id, kf_id)] = float(cs.mean())

df["vignette_score"] = df.apply(
    lambda r: vignette_scores.get((r["dataset_id"], r["keyframe_id"]), np.nan),
    axis=1
)
FEAT_COLS = [c for c in df.columns if c not in META_COLS]
print(f"Features tras vignette_score: {len(FEAT_COLS)}")
df[["dataset_id", "keyframe_id", "vignette_score"]].head(20)

### 1d. Serie temporal de luminancia desde el video

El keyframe estático es una sola imagen, pero el video `data/rgb.mp4` contiene ~100–200 frames del mismo escenario. La variabilidad temporal de luminancia revela **drift de iluminación** (el cirujano mueve el endoscopio) y picos de **especularidad** que el keyframe estático no captura.

Se extraen dos features adicionales:
- `video_lum_std`: desviación estándar de la luminancia media a lo largo del video (drift).
- `video_sat_max`: máximo de `sat_pct` a lo largo del video (peor momento de especularidad).

**Nota técnica:** el MP4 debe extraerse a disco porque `cv2.VideoCapture` no acepta buffers en memoria. Se usa `tempfile.NamedTemporaryFile` en `C:/temp` y se elimina al terminar.

In [ ]:
os.makedirs("C:/temp", exist_ok=True)

video_path_in_zip = "dataset_1/keyframe_1/data/rgb.mp4"

with zipfile.ZipFile(DATASETS["dataset_1"]["zip"], "r") as zf:
    with zf.open(video_path_in_zip) as src:
        tmp = tempfile.NamedTemporaryFile(
            suffix=".mp4", dir="C:/temp", delete=False
        )
        tmp.write(src.read())
        tmp.close()
        tmp_path = tmp.name

cap = cv2.VideoCapture(tmp_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Frames totales en el video: {total_frames}")

video_rows = []
frame_idx = 0
sample_every = 20

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % sample_every == 0:
        H_v = frame.shape[0]
        left_half = frame[: H_v // 2]               # cámara izquierda (arriba)
        left_rgb = cv2.cvtColor(left_half, cv2.COLOR_BGR2RGB)
        R = left_rgb[:, :, 0].astype(np.float32)
        G = left_rgb[:, :, 1].astype(np.float32)
        B = left_rgb[:, :, 2].astype(np.float32)
        lum = 0.299 * R + 0.587 * G + 0.114 * B
        sat = float((left_rgb > 240).any(axis=2).sum() / (R.size) * 100)
        video_rows.append(dict(
            frame_idx=frame_idx,
            lum_mean=float(lum.mean()),
            sat_pct=sat,
        ))
    frame_idx += 1

cap.release()
os.unlink(tmp_path)

df_video = pd.DataFrame(video_rows)
print(f"Frames muestreados: {len(df_video)}")
df_video.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

color_lum = "#0036a3"
color_sat = "#E0A800"

ax1.plot(df_video["frame_idx"], df_video["lum_mean"],
         color=color_lum, lw=1.5, label="Luminancia media")
ax1.set_xlabel("Frame (índice)")
ax1.set_ylabel("Luminancia media (0–255)", color=color_lum)
ax1.tick_params(axis="y", labelcolor=color_lum)

ax2 = ax1.twinx()
ax2.fill_between(df_video["frame_idx"], df_video["sat_pct"],
                 alpha=0.35, color=color_sat, label="% píxeles saturados")
ax2.set_ylabel("Saturados (%) ", color=color_sat)
ax2.tick_params(axis="y", labelcolor=color_sat)

ax1.set_title("Serie temporal de luminancia y saturación — dataset_1 / keyframe_1")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax1.text(0.5, -0.15, "Fuente: elaboración propia con base en Allan et al. (2021)",
         transform=ax1.transAxes, fontsize=8, ha="center", color="gray")
plt.tight_layout()
plt.show()

video_lum_std = float(df_video["lum_mean"].std())
video_sat_max = float(df_video["sat_pct"].max())
print(f"video_lum_std = {video_lum_std:.3f}")
print(f"video_sat_max = {video_sat_max:.3f}")

In [ ]:
# Añadir al dataframe principal (solo dataset_1/keyframe_1 tiene el video procesado;
# para el resto se usa NaN hasta procesar todos los videos)
df["video_lum_std"] = np.nan
df["video_sat_max"] = np.nan

mask = (df["dataset_id"] == "dataset_1") & (df["keyframe_id"] == "keyframe_1")
df.loc[mask, "video_lum_std"] = video_lum_std
df.loc[mask, "video_sat_max"] = video_sat_max

FEAT_COLS = [c for c in df.columns if c not in META_COLS]
print(f"Features totales tras features de video: {len(FEAT_COLS)}")
df[["dataset_id", "keyframe_id", "video_lum_std", "video_sat_max"]].head(10)

---

## Sección 2 — Normalización y transformaciones

### 2a. Cap a 150 mm

En cirugía laparoscópica con sistema da Vinci, la distancia máxima fisiológica entre el endoscopio y el tejido es de aproximadamente 150 mm; distancias mayores son físicamente implausibles y corresponden a errores del sensor de luz estructurada o a reflexiones especulares que producen valores espurios.

Godard et al. (2019) aplican un cap equivalente en KITTI (80 m); Espinosa-Loera et al. (2025) reportan que el cap a 150 mm en SCARED elimina menos del 2% de píxeles válidos y reduce el RMSE reportado.

In [ ]:
# Cargar el canal Z de dataset_1/keyframe_1 para ilustración
with zipfile.ZipFile(DATASETS["dataset_1"]["zip"], "r") as zf:
    Z_demo, _ = load_depth_and_image(zf, "dataset_1", "keyframe_1")

Z_valid = Z_demo[~np.isnan(Z_demo)]          # todos los píxeles válidos
Z_capped = np.clip(Z_valid, 0, 150)

pct_affected = float((Z_valid > 150).sum() / Z_valid.size * 100)
print(f"Píxeles afectados por el clip a 150 mm: {pct_affected:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

axes[0].hist(Z_valid, bins=80, color="#0036a3", alpha=0.8, edgecolor="white", lw=0.3)
axes[0].axvline(150, color="#E0A800", lw=2, ls="--", label="Corte 150 mm")
axes[0].set_title("Distribución de Z sin cap")
axes[0].set_xlabel("Profundidad (mm)")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

axes[1].hist(Z_capped, bins=80, color="#0036a3", alpha=0.8, edgecolor="white", lw=0.3)
axes[1].axvline(150, color="#E0A800", lw=2, ls="--", label="Corte 150 mm")
axes[1].set_title("Distribución de Z con cap a 150 mm")
axes[1].set_xlabel("Profundidad (mm)")
axes[1].set_ylabel("Frecuencia")
axes[1].legend()

for ax in axes:
    ax.text(0.5, -0.15, "Fuente: elaboración propia con base en Allan et al. (2021)",
            transform=ax.transAxes, fontsize=8, ha="center", color="gray")
plt.suptitle("Efecto del cap a 150 mm — dataset_1 / keyframe_1", y=1.02)
plt.tight_layout()
plt.show()

### 2b. Transformación logarítmica log(Z+1)

La distribución de profundidades tiene sesgo positivo: la mayoría de los píxeles válidos están entre 20 y 80 mm, pero hay una cola derecha hasta 150 mm (post-cap). La transformación log(Z+1) reduce el sesgo y comprime la escala, lo cual es coherente con el RMSE logarítmico (log RMSE) que reporta MonoIIT como métrica de evaluación.

La transformación logarítmica se aplica tanto al preprocesamiento del tensor de entrenamiento como a la columna `log_depth_median` del dataframe.

**Referencia:** Eigen et al. (2014). *Depth Map Prediction from a Single Image using a Multi-Scale Deep Network*. NIPS.

In [ ]:
Z_cap_full = np.where((Z_demo > 0) & (Z_demo < 150), Z_demo, np.nan)
Z_flat = Z_cap_full[~np.isnan(Z_cap_full)]

sk_before = skew(Z_flat)
Z_log     = np.log1p(Z_flat)
sk_after  = skew(Z_log)
reduccion = (1 - abs(sk_after) / abs(sk_before)) * 100

print(f"{'Métrica':<25} {'Antes':>10} {'Después':>10}")
print("-" * 47)
print(f"{'Skewness':<25} {sk_before:>10.3f} {sk_after:>10.3f}")
print(f"{'Reducción (%)':<25} {'':>10} {reduccion:>10.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(Z_flat, bins=80, color="#0036a3", alpha=0.8, edgecolor="white", lw=0.3)
axes[0].set_title(f"Z (mm) — sesgo = {sk_before:.3f}")
axes[0].set_xlabel("Profundidad (mm)")
axes[0].set_ylabel("Frecuencia")

axes[1].hist(Z_log, bins=80, color="#0036a3", alpha=0.8, edgecolor="white", lw=0.3)
axes[1].set_title(f"log(Z+1) — sesgo = {sk_after:.3f}")
axes[1].set_xlabel("log(Profundidad + 1)")
axes[1].set_ylabel("Frecuencia")

for ax in axes:
    ax.text(0.5, -0.15, "Fuente: elaboración propia con base en Allan et al. (2021)",
            transform=ax.transAxes, fontsize=8, ha="center", color="gray")
plt.suptitle("Transformación log(Z+1) — dataset_1 / keyframe_1", y=1.02)
plt.tight_layout()
plt.show()

### 2c. Normalización min-max del dataframe

Las features tienen unidades heterogéneas: `depth_median` está en mm, `laplacian_var` en px², `sat_pct` en %, `lum_mean` en [0,255]. Sin normalización, las features de mayor escala dominan los métodos de selección basados en distancia (correlación, varianza).

La normalización min-max escala cada feature al rango [0, 1]. **Práctica correcta de modelado:** el scaler se ajusta exclusivamente sobre los datos de train (`split == "train"`) y se aplica a todo el dataset, evitando fuga de información del conjunto de test.

In [ ]:
NUMERIC_FEATS = [c for c in FEAT_COLS
                 if df[c].dtype in [np.float64, np.float32, float]
                 and df[c].notna().sum() >= 10]

print("Features numéricas a normalizar:", NUMERIC_FEATS)
print(f"\n{'─'*60}")
print("Estadísticas ANTES de normalizar:")
print(df[NUMERIC_FEATS].describe().round(3).to_string())

# Fit solo en train
scaler = MinMaxScaler()
train_mask = df["split"] == "train"
scaler.fit(df.loc[train_mask, NUMERIC_FEATS].fillna(0))

df_scaled = df.copy()
df_scaled[NUMERIC_FEATS] = scaler.transform(df[NUMERIC_FEATS].fillna(0))

print(f"\n{'─'*60}")
print("Estadísticas DESPUÉS de normalizar (min-max):")
print(df_scaled[NUMERIC_FEATS].describe().round(3).to_string())

---

## Sección 3 — Selección y extracción de características

### 3a. Matriz de correlación de Pearson

Features altamente correlacionadas (|r| > 0.8) son redundantes: aportan la misma información y pueden causar multicolinealidad en modelos lineales. La matriz de correlación de Pearson permite identificar estos pares y decidir cuáles conservar.

En el contexto del proyecto, la correlación entre `sat_pct` y `depth_valid_pct` tiene interpretación física directa: los reflejos especulares (especularidad) producen píxeles saturados que coinciden con zonas donde el sensor de luz estructurada no puede medir profundidad correctamente.

In [ ]:
corr_feats = [c for c in NUMERIC_FEATS
              if df_scaled[c].notna().sum() == len(df_scaled)]

corr_matrix = df_scaled[corr_feats].corr(method="pearson")

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title("Correlación de Pearson — features por keyframe (escala min-max)")
ax.text(0.5, -0.08, "Fuente: elaboración propia con base en Allan et al. (2021)",
        transform=ax.transAxes, fontsize=8, ha="center", color="gray")
plt.tight_layout()
plt.show()

# Pares con |r| > 0.8
high_corr = []
for i, c1 in enumerate(corr_feats):
    for j, c2 in enumerate(corr_feats):
        if j <= i:
            continue
        r = corr_matrix.loc[c1, c2]
        if abs(r) > 0.8:
            high_corr.append((c1, c2, round(r, 3)))

print("\nPares con |r| > 0.8:")
if high_corr:
    for c1, c2, r in high_corr:
        print(f"  {c1:30s} ↔ {c2:30s}  r = {r:+.3f}")
else:
    print("  Ninguno")

**Interpretación física — `sat_pct` vs `depth_valid_pct`:**

Cuando un píxel tiene cualquier canal RGB > 240, el sensor de luz estructurada sufre sobreexposición local y no puede calcular el desfase de fase necesario para triangular la profundidad. Esto explica la correlación negativa entre `sat_pct` (reflejos especulares) y `depth_valid_pct` (cobertura del GT): más reflejos → menos profundidad medible. Esta relación es la motivación central del proyecto: eliminar los reflejos especulares aumentaría `depth_valid_pct` y mejoraría la calidad del GT disponible para entrenamiento.

### 3b. Umbral de varianza

Features con varianza cercana a cero son prácticamente constantes en todos los keyframes y no aportan información discriminativa para el modelo. `VarianceThreshold` elimina aquellas con varianza por debajo de un umbral.

**Advertencia de poder estadístico:** con 20 filas (4 datasets × 5 keyframes), las estimaciones de varianza y correlación tienen alta incertidumbre. Con los 9 datasets completos del SCARED (45 keyframes) los resultados serían más robustos. Las decisiones de descarte deben confirmarse al escalar el análisis.

In [ ]:
vt = VarianceThreshold(threshold=0.01)
vt.fit(df_scaled[corr_feats].fillna(0))

passed  = [c for c, s in zip(corr_feats, vt.get_support()) if s]
removed = [c for c, s in zip(corr_feats, vt.get_support()) if not s]

print(f"Features que PASAN el umbral de varianza (≥ 0.01): {len(passed)}")
for c in passed:
    print(f"  ✓ {c:<35} var = {df_scaled[c].var():.4f}")

print(f"\nFeatures ELIMINADAS (var < 0.01): {len(removed)}")
for c in removed:
    print(f"  ✗ {c:<35} var = {df_scaled[c].var():.4f}")

### 3c. PCA sobre la nube de puntos 3D

El análisis de componentes principales (PCA) aplicado a la nube de puntos OBJ reduce la geometría 3D del tejido a sus direcciones de mayor varianza. En geometría diferencial, estas corresponden aproximadamente a las **curvaturas principales** del tejido: la primera componente captura la extensión global de la escena, la segunda la curvatura dominante, la tercera la curvatura secundaria.

Esta técnica es relevante porque los modelos de profundidad monocular aprenden representaciones geométricas implícitas; conocer la curvatura dominante del tejido permite analizar en qué tipos de geometría el modelo comete más error.

**Referencia:** Jolliffe, I. T. (2002). *Principal Component Analysis* (2nd ed.). Springer.

In [ ]:
with zipfile.ZipFile(DATASETS["dataset_1"]["zip"], "r") as zf:
    obj_name = "dataset_1/keyframe_1/point_cloud.obj"
    with zf.open(obj_name) as f:
        lines = f.read().decode("utf-8").splitlines()

vertices = []
for line in lines:
    if line.startswith("v "):
        parts = line.split()
        vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])

pts = np.array(vertices, dtype=np.float32)
print(f"Vértices totales: {len(pts):,}")
print(f"Rango X: [{pts[:,0].min():.1f}, {pts[:,0].max():.1f}] mm")
print(f"Rango Y: [{pts[:,1].min():.1f}, {pts[:,1].max():.1f}] mm")
print(f"Rango Z: [{pts[:,2].min():.1f}, {pts[:,2].max():.1f}] mm")

rng = np.random.default_rng(42)
idx = rng.choice(len(pts), size=min(10_000, len(pts)), replace=False)
sample = pts[idx]

In [ ]:
pca = PCA(n_components=3, random_state=42)
pca.fit(sample)
coords = pca.transform(sample)

ev = pca.explained_variance_ratio_
ev_cum = np.cumsum(ev)

print(f"{'Componente':<15} {'Var. explicada':>16} {'Var. acumulada':>16}")
print("─" * 50)
for i, (e, ec) in enumerate(zip(ev, ev_cum), 1):
    print(f"PC{i:<13} {e*100:>15.2f}%  {ec*100:>15.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar([f"PC{i}" for i in range(1, 4)],
                   ev * 100, color="#0036a3", alpha=0.8, label="Var. explicada")
ax0b = axes[0].twinx()
ax0b.plot([f"PC{i}" for i in range(1, 4)], ev_cum * 100,
          "o--", color="#E0A800", lw=2, label="Acumulada")
axes[0].set_ylabel("Varianza explicada (%)")
ax0b.set_ylabel("Varianza acumulada (%)")
axes[0].set_title("Varianza explicada por componente principal")
axes[0].set_ylim(0, 110)
ax0b.set_ylim(0, 110)
lines1, labels1 = axes[0].get_legend_handles_labels()
lines2, labels2 = ax0b.get_legend_handles_labels()
axes[0].legend(lines1 + lines2, labels1 + labels2, loc="center right")

sc = axes[1].scatter(coords[:, 0], coords[:, 1],
                     c=sample[:, 2], cmap="viridis", s=3, alpha=0.6)
plt.colorbar(sc, ax=axes[1], label="Z original (mm)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("Proyección PC1 vs PC2 (color = profundidad Z)")

for ax in axes:
    ax.text(0.5, -0.12, "Fuente: elaboración propia con base en Allan et al. (2021)",
            transform=ax.transAxes, fontsize=8, ha="center", color="gray")
plt.suptitle("PCA sobre nube de puntos 3D — dataset_1 / keyframe_1", y=1.02)
plt.tight_layout()
plt.show()

**Interpretación geométrica:**

- **PC1** concentra la mayor parte de la varianza y corresponde a la **extensión global de la escena**: captura si el tejido está cerca o lejos del endoscopio. Una alta varianza en PC1 indica escenas con gran variación de distancia (geometría compleja).
- **PC2** captura la **curvatura principal**: la dirección en la que el tejido se dobla o arquea más. Un peso alto en PC2 indica pliegues o estructuras anatómicas con curvatura pronunciada.
- **PC3** captura la **curvatura secundaria** ortogonal a PC2. Su baja varianza indica que la mayoría de la geometría puede describirse bien en el plano PC1–PC2.

El scatter coloreado por Z muestra que PC1 está fuertemente correlacionado con la profundidad original, lo cual es coherente con su interpretación como extensión de la escena.

---

## Sección 4 — Conclusión CRISP-ML(Q)

### Artefactos producidos en esta fase

Esta fase de ingeniería de características produjo los siguientes artefactos concretos para la siguiente etapa del proyecto:

1. **Dataframe de 20 filas × N features**: una representación tabular derivada del dataset SCARED (4 datasets × 5 keyframes) con features estadísticas extraídas de la imagen RGB y del mapa de profundidad TIFF. Este dataframe es el insumo para el diagnóstico de calidad de datos, la selección de features y el análisis de correlaciones presentados en este notebook.

2. **Pipeline de preprocesamiento de tensores**: las transformaciones definidas aquí — cap a 150 mm, transformación log(Z+1), normalización min-max ajustada sobre train — se reutilizarán directamente en el training loop de los modelos NeXF, EndoGaussian y EndoDepthAndMotion. Esto garantiza coherencia entre el preprocesamiento analítico y el preprocesamiento de entrenamiento.

3. **Máscara de viñeteado gaussiano**: el ajuste gaussiano sobre el perfil radial de validez del GT permite cuantificar el efecto del viñeteado en cada keyframe mediante `vignette_score`. Keyframes con `vignette_score` bajo son aquellos donde el viñeteado del endoscopio afecta más la cobertura del GT, y por tanto son los más beneficiados por la corrección de iluminación propuesta.

4. **Perfil de varianza y correlación entre features**: la matriz de correlación de Pearson y el umbral de varianza identifican features redundantes y de baja información que no se incluirán como inputs adicionales en el modelado.

### Decisiones informadas para la fase de modelado

El análisis de esta fase permite tomar las siguientes decisiones para la siguiente etapa:

- **Selección de keyframes de referencia**: los keyframes con mayor `depth_valid_pct` y menor `sat_pct` son los mejores candidatos para el análisis inicial del pipeline, ya que ofrecen más GT disponible y menos contaminación por reflejos especulares.
- **Preprocesamiento estándar**: el cap a 150 mm y el log(Z+1) se aplicarán en todas las iteraciones del training loop. La normalización min-max se aplicará siempre ajustando el scaler sobre datos de train exclusivamente.
- **Features descartadas por redundancia**: las features identificadas con |r| > 0.8 o varianza < 0.01 no se incluirán como inputs adicionales en modelos de diagnóstico secundarios, para evitar multicolinealidad y reducir la dimensionalidad del espacio de features.

### Alineación con CRISP-ML(Q)

Esta fase cierra el ciclo de **Preparación de Datos** del marco CRISP-ML(Q) (Visengeriyeva et al., 2023). Los artefactos producidos son los inputs directos de la siguiente fase de **Modelado**: el dataframe procesado alimenta el análisis de selección de modelos, y el pipeline de preprocesamiento de tensores se integra directamente en el loop de entrenamiento.

El criterio de calidad Q aplicado en esta fase incluye: trazabilidad de cada transformación, separación explícita de train/test en el scaler, y documentación de las limitaciones estadísticas derivadas del tamaño reducido del subconjunto analizado (20 keyframes de los 45 disponibles en el dataset completo).

**Referencia:** Visengeriyeva, L., et al. (2023). *CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology*. https://ml-ops.org/content/crisp-ml

---

## Resumen final del notebook

In [ ]:
FEAT_COLS_FINAL = [c for c in df.columns if c not in META_COLS]

print("=" * 60)
print(f"Archivo: Avance2.52")
print(f"Shape del dataframe final: {df.shape}")
print(f"{'=' * 60}")
print(f"\n{'Feature':<30} {'Tipo':<15} {'No-nulos':>10}")
print("-" * 57)
for col in FEAT_COLS_FINAL:
    dtype = str(df[col].dtype)
    non_null = df[col].notna().sum()
    print(f"{col:<30} {dtype:<15} {non_null:>10}")
print(f"\nMetadatos: {META_COLS}")
print(f"\nTotal features numéricas: {len(FEAT_COLS_FINAL)}")